# Importing libraries

In [56]:
import pandas as pd
import graphviz
from sklearn.tree import export_graphviz
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score
from sklearn.metrics import f1_score,confusion_matrix,classification_report

# Loading file data into Data-frame

In [57]:
df = pd.read_csv("/home/umair-nawaz/Downloads/test/german.data",sep=" ",header=None)

# Preprocessing

In [58]:
# Gibving columns to a meaningfull name  collected thorugh the dataset.doc
df.columns = ["Chekcing_amount","Duration(months)","Credit_history","Purpose"
              ,"Credit_amount","Saving_acount","Employment_since",
              "Installment_rate","Personalstatus_Sex","Guarantors",
              "Residence_since","Property","Age","installment_plan",
              "Housing","existing_credits","Job","People_liable","Telephone",
              "forigen_worker","target"]
df

,Chekcing_amount,Duration(months),Credit_history,Purpose,Credit_amount,Saving_acount,Employment_since,Installment_rate,Personalstatus_Sex,Guarantors,...,Property,Age,installment_plan,Housing,existing_credits,Job,People_liable,Telephone,forigen_worker,target
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,A14,12,A32,A42,1736,A61,A74,3,A92,A101,...,A121,31,A143,A152,1,A172,1,A191,A201,1
996,A11,30,A32,A41,3857,A61,A73,4,A91,A101,...,A122,40,A143,A152,1,A174,1,A192,A201,1
997,A14,12,A32,A43,804,A61,A75,4,A93,A101,...,A123,38,A143,A152,1,A173,1,A191,A201,1
998,A11,45,A32,A43,1845,A61,A73,4,A93,A101,...,A124,23,A143,A153,1,A173,1,A192,A201,2


In [59]:
df["target"].value_counts()

target
1    700
2    300
Name: count, dtype: int64

In [60]:
# Mapping two categorical Column into 0's and 1's
df["target"] = df["target"].map({1:0,2:1}) # after ,apping 1=able 2=not able
df["forigen_worker"] = df["forigen_worker"].map({"A201":1 , "A202":0})
df["Telephone"] = df["Telephone"].map({"A191":0 , "A192":1})

In [61]:
#  Testing is any null values in data or is it clean ?
df.isnull().sum()

Chekcing_amount       0
Duration(months)      0
Credit_history        0
Purpose               0
Credit_amount         0
Saving_acount         0
Employment_since      0
Installment_rate      0
Personalstatus_Sex    0
Guarantors            0
Residence_since       0
Property              0
Age                   0
installment_plan      0
Housing               0
existing_credits      0
Job                   0
People_liable         0
Telephone             0
forigen_worker        0
target                0
dtype: int64

# Separating Input and Output Data

In [62]:
X = df.drop("target",axis=1)
y = df["target"]

# Spilting Data into training anf testing

In [63]:
x_train,x_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

# Applying One Hot Encoding on all input columns (x only) to avoid data Leakage

In [64]:
ohe = OneHotEncoder(drop="first",sparse_output=False)
categorical_columns = ["Job","Housing","installment_plan","Property",
                                "Guarantors","Personalstatus_Sex","Employment_since",
                                "Saving_acount","Purpose","Credit_history",
                                "Chekcing_amount"]
x_train_encoded = ohe.fit_transform(x_train[categorical_columns])
x_test_encoded = ohe.transform(x_test[categorical_columns])

# Converting Bencoded data back to dataframe

In [65]:
x_train_encoded = pd.DataFrame(x_train_encoded, 
                          columns=ohe.get_feature_names_out(),
                          index=x_train.index)
x_test_encoded = pd.DataFrame(x_test_encoded ,
                          columns=ohe.get_feature_names_out(),
                          index=x_test.index)

# Concating the encoded dataframe  back to input x_train and x_test

In [66]:
x_train = pd.concat([x_train,x_train_encoded],axis=1)
x_test = pd.concat([x_test,x_test_encoded],axis=1)

# Droping the orignal columns on which we apply encoding

In [67]:
x_test.drop(columns=categorical_columns,inplace=True)
x_train.drop(columns=categorical_columns,inplace=True)

# Testing model by applyingg various depths and checking its results how it performs

In [68]:
# With imbalanced weights see how it works
for i in range(4,25):
    model = DecisionTreeClassifier(max_depth=i,random_state=42)
    model.fit(x_train,y_train)
    y_pred = model.predict(x_test)
    recall = recall_score(y_test,y_pred)
    precision = precision_score(y_test,y_pred)
    f1 = f1_score(y_test,y_pred)
    accuracy = accuracy_score(y_test,y_pred)
    print("\n\nDepth: ",i)
    print("Accuracy: ",accuracy)
    print("Precision: ",precision)
    print("Recall: ",recall)
    print("F1: ",f1)



Depth:  4
Accuracy:  0.695
Precision:  0.4666666666666667
Recall:  0.11666666666666667
F1:  0.18666666666666668


Depth:  5
Accuracy:  0.69
Precision:  0.4444444444444444
Recall:  0.13333333333333333
F1:  0.20512820512820512


Depth:  6
Accuracy:  0.68
Precision:  0.46774193548387094
Recall:  0.48333333333333334
F1:  0.47540983606557374


Depth:  7
Accuracy:  0.685
Precision:  0.47692307692307695
Recall:  0.5166666666666667
F1:  0.496


Depth:  8
Accuracy:  0.67
Precision:  0.45714285714285713
Recall:  0.5333333333333333
F1:  0.49230769230769234


Depth:  9
Accuracy:  0.645
Precision:  0.4153846153846154
Recall:  0.45
F1:  0.432


Depth:  10
Accuracy:  0.645
Precision:  0.42857142857142855
Recall:  0.55
F1:  0.48175182481751827


Depth:  11
Accuracy:  0.63
Precision:  0.4027777777777778
Recall:  0.48333333333333334
F1:  0.4393939393939394


Depth:  12
Accuracy:  0.63
Precision:  0.4027777777777778
Recall:  0.48333333333333334
F1:  0.4393939393939394


Depth:  13
Accuracy:  0.625
Prec

In [69]:
# Now makes both classes(weights) balanced so it amy increase in some perfomance
for i in range(4,25):
    model = DecisionTreeClassifier(max_depth=i,class_weight="balanced",random_state=42)
    model.fit(x_train,y_train)
    y_pred = model.predict(x_test)
    recall = recall_score(y_test,y_pred)
    precision = precision_score(y_test,y_pred)
    f1 = f1_score(y_test,y_pred)
    accuracy = accuracy_score(y_test,y_pred)
    print("\n\nDepth: ",i)
    print("Accuracy: ",accuracy)
    print("Precision: ",precision)
    print("Recall: ",recall)
    print("F1: ",f1)



Depth:  4
Accuracy:  0.63
Precision:  0.43
Recall:  0.7166666666666667
F1:  0.5375


Depth:  5
Accuracy:  0.64
Precision:  0.4387755102040816
Recall:  0.7166666666666667
F1:  0.5443037974683544


Depth:  6
Accuracy:  0.61
Precision:  0.41346153846153844
Recall:  0.7166666666666667
F1:  0.524390243902439


Depth:  7
Accuracy:  0.645
Precision:  0.43529411764705883
Recall:  0.6166666666666667
F1:  0.5103448275862069


Depth:  8
Accuracy:  0.705
Precision:  0.5066666666666667
Recall:  0.6333333333333333
F1:  0.562962962962963


Depth:  9
Accuracy:  0.72
Precision:  0.5285714285714286
Recall:  0.6166666666666667
F1:  0.5692307692307692


Depth:  10
Accuracy:  0.665
Precision:  0.4567901234567901
Recall:  0.6166666666666667
F1:  0.524822695035461


Depth:  11
Accuracy:  0.66
Precision:  0.44594594594594594
Recall:  0.55
F1:  0.4925373134328358


Depth:  12
Accuracy:  0.695
Precision:  0.4927536231884058
Recall:  0.5666666666666667
F1:  0.5271317829457365


Depth:  13
Accuracy:  0.675
Prec

# Now select best model using Grid-Search

In [70]:
model =  DecisionTreeClassifier(class_weight="balanced",random_state=42)

In [71]:
grid_pram = {
    "max_depth": [6,8,9,10],
    "min_samples_split": [2,5,10,20],
    "min_samples_leaf": [1,2,5,10]
}
scr = {
    "recall": "recall",
    "accuracy": "accuracy"
}

In [72]:
grid = GridSearchCV(
    model,
    grid_pram,
    scoring=scr,
    refit="recall",
    cv=5
)

In [73]:
grid.fit(x_train,y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",DecisionTreeC...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [6, 8, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'accuracy': 'accuracy', 'recall': 'recall'}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'recall'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each 

In [19]:
print(grid.best_params_)

{'max_depth': 6, 'min_samples_leaf': 5, 'min_samples_split': 20}


# Now after getting best params now train model

In [20]:
model = DecisionTreeClassifier(
    max_depth=6,
    min_samples_split=20,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
)

In [21]:
model.fit(x_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",6
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",20
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current n

# Prediction

In [22]:
y_pred = model.predict(x_test)

In [23]:
y_pred

array([1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1,
       0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0,
       0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0,
       1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0,
       1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1,
       1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1,
       0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0,
       0, 1])

In [24]:
y_test

30     0
128    0
289    1
216    0
966    1
      ..
522    1
977    0
52     0
542    1
423    0
Name: target, Length: 200, dtype: int64

# Final Accuracy

In [25]:
print("Accuracy: ", accuracy_score(y_test,y_pred))

Accuracy:  0.64


# Final Precision

In [26]:
print("Precision: ", precision_score(y_test,y_pred))

Precision:  0.43617021276595747


# Final Recall

In [27]:
print("Recall: ", recall_score(y_test,y_pred))

Recall:  0.6833333333333333


# Final F1

In [28]:
print("F1:", f1_score(y_test, y_pred))

F1: 0.5324675324675324


# Final Confusion matrix

In [29]:
confusion_matrix(y_test, y_pred)

array([[87, 53],
       [19, 41]])

# Clasclassification_report

In [30]:
classification_report(y_test, y_pred)

'              precision    recall  f1-score   support\n\n           0       0.82      0.62      0.71       140\n           1       0.44      0.68      0.53        60\n\n    accuracy                           0.64       200\n   macro avg       0.63      0.65      0.62       200\nweighted avg       0.71      0.64      0.65       200\n'

In [31]:
dot_data = export_graphviz(
    model,
    out_file=None,
    feature_names=x_train.columns,
    class_names=["Non-Defaulter", "Defaulter"],
    filled=True,
    rounded=True,
    special_characters=True
)

In [32]:
graph.render("loan_default_graph")

NameError: name 'graph' is not defined

In [33]:
Y_training_pred = model.predict(x_train)
Y_testing_pred = model.predict(x_test)

In [35]:
testing_accuracy = accuracy_score(y_test,Y_testing_pred)
training_accuracy = accuracy_score(y_train,Y_training_pred)
print("Training Accuracy: " , training_accuracy)
print("Testing Accuracy:  " , testing_accuracy)

Training Accuracy:  0.76625
Testing Accuracy:   0.64
